# 📚 Multiclass Text Classification with Neural Networks
This notebook demonstrates how to solve a multiclass classification problem using text data with both **TensorFlow/Keras** and **PyTorch**.

## 📦 Install Dependencies

In [ ]:
# Uncomment if running in Colab
# !pip install -q tensorflow torch scikit-learn pandas


## 📂 Load and Prepare the 20 Newsgroups Dataset

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Load text and labels
newsgroups = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))
texts = newsgroups.data
labels = newsgroups.target
label_names = newsgroups.target_names

# Split
X_train_text, X_test_text, y_train, y_test = train_test_split(texts, labels, test_size=0.2, random_state=42)

# TF-IDF vectorization
vectorizer = TfidfVectorizer(max_features=2000)
X_train = vectorizer.fit_transform(X_train_text).toarray()
X_test = vectorizer.transform(X_test_text).toarray()

# Encode labels for PyTorch
num_classes = np.unique(y_train).shape[0]
print(f"Classes: {num_classes}, Shape: {X_train.shape}")


## 🧠 Keras (TensorFlow) Implementation

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Convert labels to categorical
y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes)
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes)

model = Sequential([
    Dense(512, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(256, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train_cat, epochs=5, batch_size=64, validation_split=0.1)


### 🔍 Evaluate Keras Model

In [ ]:
loss, acc = model.evaluate(X_test, y_test_cat)
print(f"Keras Test Accuracy: {acc:.3f}")


## 🔁 PyTorch Implementation

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Prepare PyTorch data
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_ds = TensorDataset(X_train_tensor, y_train_tensor)
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)

# Define model
class TextNet(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.net(x)

model_torch = TextNet(X_train.shape[1], num_classes)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_torch.parameters(), lr=0.001)


### 🔧 Train PyTorch Model

In [ ]:
for epoch in range(5):
    for xb, yb in train_dl:
        preds = model_torch(xb)
        loss = loss_fn(preds, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


### 🔍 Evaluate PyTorch Model

In [ ]:
with torch.no_grad():
    preds = model_torch(X_test_tensor)
    predicted = torch.argmax(preds, dim=1)
    accuracy = (predicted == y_test_tensor).float().mean()
    print(f"PyTorch Test Accuracy: {accuracy:.3f}")
